# 18 · Model 1 profile audit
Owner: Disha. Checks a finished profile run before Ruchika trains on it. Run it on the **pilot** now
(`pilot_v2_qwen35_9b`) and on the **full run** when it ends (`full_v2_qwen35_9b`): change one setting.
CPU is enough (Colab CPU runtime): about 5 minutes for the pilot, 15–25 minutes for the full run.

### What it checks
| Group | Checks | If it fails |
|---|---|---|
| **Files** | every file listed in `checksums.json` is there and unchanged | FAIL |
| **Format** | each `.npz` loads with Ruchika's `read_profile_file` (copied in verbatim); rows match `user_profiles.parquet` | FAIL |
| **Coverage** | profiles + failed users = every user that should be there (all 56,017, or the 200 pilot users); no overlap | FAIL |
| **Empty / duplicate** | no empty profiles; duplicate texts and near-duplicate vectors between users with *different* inputs | empty: FAIL, duplicates: WARN |
| **Training data only** | rebuilt **independently** from `interactions.parquet` (its own timestamps, not `sequences.parquet`, not the `train_items_only` flag): every user's inputs are training rows, never their validation or test item, and the rebuilt input hash equals the stored `input_sha256` | FAIL |
| **Text leakage** | how often a profile names a brand that appears only in the user's held-out (validation/test) items, compared with a random-user baseline | WARN if clearly above chance |
| **Vectors match texts** | a sample of profiles is re-embedded; each stored vector must match its own text | FAIL |
| **Settings** | approved prompt, pinned model commit, thinking off | FAIL |
| **Generation hygiene** | thinking text, reasoning or meta text, very long profiles, possible sensitive terms | WARN |
| **Chunks** (full run) | the per-part files given to Ruchika early agree with the final files | FAIL |

### Output (the profile folder itself is never changed)
`298A/model1_profiles/audits/<run name>/audit_report.json` and `audit_summary.md`

In [1]:
# ============================== SETTINGS ==============================
RUN_NAME = 'full_v2_qwen35_9b'     # after the full run: 'full_v2_qwen35_9b'
PROJECT_ROOT = None                 # the shared '298A' folder; None = find it (Colab: /content/drive/MyDrive/298A)
EXPECTED_PROMPT_SHA256 = 'a18472f1aefb1a5f3b6397d20d6dc2335a382a6b69c681b908e3e23ba97e9727'
EXPECTED_TEACHER = 'Qwen/Qwen3.5-9B'
EXPECTED_TEACHER_REVISION = 'c202236235762e1c871ad0ccb60c8ee5ba337b9a'
ENCODER_IDS = {'minilm': 'sentence-transformers/all-MiniLM-L6-v2', 'e5_large': 'intfloat/multilingual-e5-large'}
ENCODER_REVISIONS = {'minilm': '1110a243fdf4706b3f48f1d95db1a4f5529b4d41', 'e5_large': '3d7cfbdacd47fdda877c5cd8a79fbcc4f2a574f3'}
REEMBED_SAMPLE = 100                # profiles re-embedded per encoder to check vectors match their texts
NEAR_DUP_COSINE = 0.98              # MiniLM cosine above which two different users' profiles count as near-duplicates
N_USERS_FULL = 56017
PILOT_STRATA = [(3, 4, 100), (5, 9, 60), (10, None, 40)]   # the pilot's selection rule, to rebuild its user list
SEED = 42
# ======================================================================

In [2]:
import sys, subprocess, importlib.metadata as md
def _ver(name):
    try:
        return md.version(name)
    except md.PackageNotFoundError:
        return None
PINNED = {'transformers': '5.24.0', 'sentence-transformers': '6.3.0'}
need = [f'{n}=={v}' for n, v in PINNED.items() if _ver(n) != v]
if need:
    print('Installing', need, '...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *need], check=True)
    if any(m in sys.modules for m in ('transformers', 'sentence_transformers', 'huggingface_hub', 'tokenizers')):
        raise RuntimeError('Installed new library versions. RESTART the runtime/kernel now, then Run all again.')
print({n: _ver(n) for n in ('transformers', 'sentence-transformers', 'numpy', 'pandas', 'pyarrow')})

Installing ['transformers==5.24.0', 'sentence-transformers==6.3.0'] ...
{'transformers': '5.24.0', 'sentence-transformers': '6.3.0', 'numpy': '2.4.6', 'pandas': '2.2.3', 'pyarrow': '23.0.1'}


## 1. Locations

In [3]:
import os, json, re, glob, hashlib, datetime, random
from pathlib import Path
import numpy as np
import pandas as pd

for k in ('PROJECT_ROOT', 'RUN_NAME', 'EXPECTED_TEACHER', 'EXPECTED_TEACHER_REVISION'):          # local testing only
    if os.environ.get(f'AUDIT_{k}'):
        globals()[k] = os.environ[f'AUDIT_{k}']
for k in ENCODER_IDS:
    if os.environ.get(f'AUDIT_ENCODER_{k.upper()}'):
        ENCODER_IDS[k] = os.environ[f'AUDIT_ENCODER_{k.upper()}']
    if os.environ.get(f'AUDIT_ENCODER_REVISION_{k.upper()}'):
        ENCODER_REVISIONS[k] = os.environ[f'AUDIT_ENCODER_REVISION_{k.upper()}']
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

DATA_REL = Path('04_Data') / 'NewDataRunFiles' / 'processed'
def find_project_root():
    home = Path.home()
    bases = [Path('/content/drive/MyDrive'), *Path('/content/drive/Shareddrives').glob('*')]
    bases += [Path(f'{d}:/') / sub for d in 'DEFGHIJKLMNOPQRSTUVWXYZ' for sub in ('My Drive', 'Shared drives/*')]
    bases += [*home.glob('Library/CloudStorage/GoogleDrive-*/My Drive'), *home.glob('Library/CloudStorage/GoogleDrive-*/Shared drives/*'),
              home / 'Google Drive' / 'My Drive', home / 'GoogleDrive', home / 'gdrive', home / 'Google Drive', home]
    found = []
    for b in bases:
        for base in (glob.glob(str(b)) if '*' in str(b) else [str(b)]):
            for cand in (Path(base) / '298A', *Path(base).glob('*/298A')):
                try:
                    if (cand / DATA_REL / 'interactions.parquet').exists():
                        found.append(cand.resolve())
                except OSError:
                    pass
    found = sorted(set(found))
    if len(found) > 1 and len({(r / DATA_REL / 'interactions.parquet').stat().st_size for r in found}) == 1:
        found = found[:1]
    return found

if PROJECT_ROOT is None:
    roots = find_project_root()
    if len(roots) != 1:
        raise FileNotFoundError('Could not find the shared 298A folder (found: ' + (', '.join(map(str, roots)) or 'none') +
                                '). Set PROJECT_ROOT in the settings cell.')
    PROJECT_ROOT = roots[0]
PROJECT_ROOT = Path(PROJECT_ROOT)
DATA_DIR = PROJECT_ROOT / DATA_REL
RUN_DIR = PROJECT_ROOT / 'model1_profiles' / RUN_NAME
assert (RUN_DIR / 'profiles_meta.json').exists(), f'no finished profile run in {RUN_DIR} (profiles_meta.json missing)'
AUDIT_DIR = PROJECT_ROOT / 'model1_profiles' / 'audits' / RUN_NAME
AUDIT_DIR.mkdir(parents=True, exist_ok=True)
META = json.loads((RUN_DIR / 'profiles_meta.json').read_text())
MODE = META.get('mode', 'pilot')
print('298A folder :', PROJECT_ROOT)
print('Profile run :', RUN_DIR, f'(mode: {MODE})')
print('Audit output:', AUDIT_DIR)

RESULTS = []                        # one row per check: group, check, status (PASS/WARN/FAIL), detail
def record(group, check, ok, detail='', warn_only=False):
    status = 'PASS' if ok else ('WARN' if warn_only else 'FAIL')
    RESULTS.append({'group': group, 'check': check, 'status': status, 'detail': str(detail)})
    print(f'[{status}] {group}: {check}' + (f' ({detail})' if detail != '' else ''))

Mounted at /content/drive
298A folder : /content/drive/.shortcut-targets-by-id/1K2gvVTjU-77GWGMI14ixFt5F57IaijZB/298A
Profile run : /content/drive/.shortcut-targets-by-id/1K2gvVTjU-77GWGMI14ixFt5F57IaijZB/298A/model1_profiles/full_v2_qwen35_9b (mode: full)
Audit output: /content/drive/.shortcut-targets-by-id/1K2gvVTjU-77GWGMI14ixFt5F57IaijZB/298A/model1_profiles/audits/full_v2_qwen35_9b


## 2. Files: everything in `checksums.json` is present and unchanged

In [4]:
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1 << 20), b''):
            h.update(block)
    return h.hexdigest()

required = ['user_profiles.parquet', 'failed_users.json', 'profiles_meta.json', 'profiles_minilm.npz', 'profiles_e5_large.npz']
missing = [f for f in required if not (RUN_DIR / f).exists()]
record('Files', 'required files present', not missing, missing or 'all 5')
if (RUN_DIR / 'checksums.json').exists():
    listed = json.loads((RUN_DIR / 'checksums.json').read_text())
    bad = [rel for rel, h in listed.items() if not (RUN_DIR / rel).exists() or sha256_file(RUN_DIR / rel) != h]
    record('Files', 'checksums.json matches every listed file', not bad, f'{len(listed) - len(bad)}/{len(listed)} match' + (f'; differ: {bad[:5]}' if bad else ''))
else:
    record('Files', 'checksums.json present', False, 'missing', warn_only=True)

[PASS] Files: required files present (all 5)
[PASS] Files: checksums.json matches every listed file (94/94 match)


## 3. Format: Ruchika's loader, copied in verbatim from `src/data/profile_file.py`

In [5]:
"""(copied verbatim from src/data/profile_file.py) File format for per-user profile embeddings (the distillation targets' raw material).

A profile file is a NumPy .npz written with write_profile_file():
    user_idx          int64 (n,)     unique user ids, 0 <= user_idx < n_users
    embeddings        float (n, d)   one text embedding per user, finite
    source            str            'llm_profile' | 'item_text_centroid' | 'synthetic'
    train_items_only  bool           the producer attests that every profile was built from TRAINING items only
                                     (no validation or test item, no review written after the user's cut-off)
Users without a row simply have no distillation term (subset ablation). NumPy only, so it is easy to produce on Colab.
"""
import hashlib
from pathlib import Path

import numpy as np

SOURCES = ('llm_profile', 'item_text_centroid', 'synthetic')


def file_sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()


def _scalar_bool(value, name):
    """Accept only a real scalar boolean: a Python bool, a NumPy bool, or a 0-d bool array.

    Strings ('False', '0'), numbers (0, 1, 2) and non-scalar arrays are rejected BEFORE any conversion, because Python
    truthiness would read 'False' and 2 as True. This validates the declaration, not the upstream contents.
    """
    if isinstance(value, np.ndarray):
        if value.ndim != 0 or value.dtype != np.bool_:
            raise ValueError(f'{name} must be a scalar boolean, not an array of dtype {value.dtype} and shape {value.shape}')
        return bool(value)
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    raise ValueError(f'{name} must be a scalar boolean (True), got {type(value).__name__}: {value!r}')


def write_profile_file(path, user_idx, embeddings, source, train_items_only):
    train_only = _scalar_bool(train_items_only, 'train_items_only')
    if not train_only:
        raise ValueError('refusing to write a profile file that is not built from training items only')
    user_idx = np.asarray(user_idx, dtype=np.int64)
    embeddings = np.asarray(embeddings, dtype=np.float32)
    np.savez(path, user_idx=user_idx, embeddings=embeddings, source=np.array(source),
             train_items_only=np.array(True))


def read_profile_file(path, *, allow_synthetic=False):
    """Load and validate a profile file; raises ValueError with a specific reason instead of returning bad data."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f'profile file not found: {path}')
    with np.load(path, allow_pickle=False) as z:
        missing = [k for k in ('user_idx', 'embeddings', 'source', 'train_items_only') if k not in z.files]
        if missing:
            raise ValueError(f'profile file is missing {missing}')
        user_idx, embeddings = z['user_idx'], z['embeddings']
        raw_source = z['source']
        if raw_source.ndim != 0 or raw_source.dtype.kind != 'U':
            raise ValueError('source must be a scalar string')
        source = str(raw_source)
        train_only = _scalar_bool(z['train_items_only'], 'train_items_only')       # strict: no truthiness
    if source not in SOURCES:
        raise ValueError(f'source must be one of {SOURCES}, got {source!r}')
    if source == 'synthetic' and not allow_synthetic:
        raise ValueError('synthetic profiles are for smoke tests only and cannot be used for a real run')
    if not train_only:
        raise ValueError('profiles must be built from training items only (train_items_only is not True)')
    if user_idx.ndim != 1 or not np.issubdtype(user_idx.dtype, np.integer):
        raise ValueError('user_idx must be a 1-D integer array')
    if embeddings.ndim != 2 or len(embeddings) != len(user_idx):
        raise ValueError('embeddings must be (n, d) with one row per user_idx')
    if len(user_idx) == 0:
        raise ValueError('profile file has no rows')
    if user_idx.min() < 0 or len(np.unique(user_idx)) != len(user_idx):
        raise ValueError('user_idx must be unique and non-negative')
    if not np.isfinite(embeddings).all():
        raise ValueError('embeddings contain non-finite values')
    return {'user_idx': user_idx.astype(np.int64), 'embeddings': embeddings.astype(np.float64), 'source': source,
            'sha256': file_sha256(path)}

In [6]:
profiles = pd.read_parquet(RUN_DIR / 'user_profiles.parquet')
failed = json.loads((RUN_DIR / 'failed_users.json').read_text())
failed_ids = {int(f['user_idx']) for f in failed}
need_cols = {'user_idx', 'profile_text', 'teacher_model', 'prompt_version', 'n_train_items_used', 'input_sha256'}
record('Format', 'user_profiles.parquet has the agreed columns', need_cols <= set(profiles.columns),
       sorted(need_cols - set(profiles.columns)) or 'ok')
record('Format', 'user_idx unique in user_profiles.parquet', profiles.user_idx.is_unique)
VEC = {}
for name, dim in (('minilm', 384), ('e5_large', 1024)):
    try:
        loaded = read_profile_file(RUN_DIR / f'profiles_{name}.npz')
        ok = loaded['source'] == 'llm_profile' and np.array_equal(loaded['user_idx'], profiles.user_idx.to_numpy())
        record('Format', f'profiles_{name}.npz loads with read_profile_file; rows match user_profiles.parquet', ok,
               f"{loaded['embeddings'].shape}, source {loaded['source']}")
        if os.environ.get('AUDIT_ANY_DIM') != '1':
            record('Format', f'profiles_{name}.npz has {dim} dimensions', loaded['embeddings'].shape[1] == dim,
                   loaded['embeddings'].shape[1])
        norms = np.linalg.norm(loaded['embeddings'], axis=1)
        record('Format', f'profiles_{name}.npz vectors are unit length', bool(np.allclose(norms, 1, atol=1e-3)),
               f'norm {norms.min():.4f}-{norms.max():.4f}')
        VEC[name] = loaded['embeddings'].astype(np.float32)
    except Exception as e:
        record('Format', f'profiles_{name}.npz loads with read_profile_file', False, f'{type(e).__name__}: {e}')

[PASS] Format: user_profiles.parquet has the agreed columns (ok)
[PASS] Format: user_idx unique in user_profiles.parquet
[PASS] Format: profiles_minilm.npz loads with read_profile_file; rows match user_profiles.parquet ((56017, 384), source llm_profile)
[PASS] Format: profiles_minilm.npz has 384 dimensions (384)
[PASS] Format: profiles_minilm.npz vectors are unit length (norm 1.0000-1.0000)
[PASS] Format: profiles_e5_large.npz loads with read_profile_file; rows match user_profiles.parquet ((56017, 1024), source llm_profile)
[PASS] Format: profiles_e5_large.npz has 1024 dimensions (1024)
[PASS] Format: profiles_e5_large.npz vectors are unit length (norm 1.0000-1.0000)


## 4. Rebuild every user's input from `interactions.parquet` alone
Ordering comes from the interactions file's own `timestamp` and `pos` columns, so this does not reuse
`sequences.parquet`, the stored item lists, or any flag. Each user's input is the last
`max_items_per_user` **training** rows (oldest to newest) with their training ratings, hashed the same way as at
generation time.

In [7]:
EXPECTED_DATA_SHA256 = {
    'interactions.parquet': '298cab581ac90b44504d512b6c706f4616ea93affa8e601be58158a8252c4905',
    'sequences.parquet': '0a55f2c375c06594b322d1e23dcd49d975ac7f8e1f7db12e4cc8bea59128186a',
    'item_text.parquet': '39dbd1ac817df3383def7c677fca830aaf36cfe470e1c0612c543486e6b8909d',
}
if os.environ.get('AUDIT_EXPECTED_HASHES_JSON'):                          # local testing with synthetic data only
    EXPECTED_DATA_SHA256 = json.loads(Path(os.environ['AUDIT_EXPECTED_HASHES_JSON']).read_text())
data_bad = [n for n, h in EXPECTED_DATA_SHA256.items() if sha256_file(DATA_DIR / n) != h]
record('Training data only', 'frozen data files match the manifest', not data_bad, data_bad or 'ok')
meta_hashes = META.get('inputs', {}).get('sha256', {})
missing_in_run = sorted(set(EXPECTED_DATA_SHA256) - set(meta_hashes))
wrong_in_run = sorted(n for n, h in EXPECTED_DATA_SHA256.items() if n in meta_hashes and meta_hashes[n] != h)
record('Training data only', 'the run recorded every expected input file with the frozen hash',
       not missing_in_run and not wrong_in_run,
       f'missing {missing_in_run}, different {wrong_in_run}' if missing_in_run or wrong_in_run
       else {n: meta_hashes[n][:12] for n in EXPECTED_DATA_SHA256})

inter = pd.read_parquet(DATA_DIR / 'interactions.parquet', columns=['user_idx', 'item_idx', 'rating', 'timestamp', 'pos', 'split'])
inter = inter.astype({'user_idx': 'int64', 'item_idx': 'int64', 'rating': 'float64'})
MAX_ITEMS = int(META.get('max_items_per_user', 50))
train = inter[inter.split == 'train'].sort_values(['user_idx', 'timestamp', 'pos'], kind='stable')
train = train.groupby('user_idx', sort=False).tail(MAX_ITEMS)
rebuilt = train.groupby('user_idx', sort=False).agg(item_list=('item_idx', list), rating_list=('rating', list))
held_out = inter[inter.split != 'train'].groupby('user_idx').item_idx.apply(set)

def input_hash(item_ids, ratings):
    return hashlib.sha256(json.dumps({'items': [int(i) for i in item_ids], 'ratings': [float(r) for r in ratings]},
                                     separators=(',', ':')).encode()).hexdigest()
p = profiles.set_index('user_idx')
has_rebuild = p.index.isin(rebuilt.index)
record('Training data only', 'every profiled user has training rows', bool(has_rebuild.all()), f'{int((~has_rebuild).sum())} without')
re_hash = pd.Series({u: input_hash(rebuilt.item_list[u], rebuilt.rating_list[u]) for u in p.index[has_rebuild]})
match = (re_hash == p.input_sha256.reindex(re_hash.index))
record('Training data only', 'rebuilt input hash equals the stored input_sha256 for every profile',
       bool(match.all()) and len(match) == len(p), f'{int(match.sum()):,}/{len(p):,} match')
n_used_ok = (p.n_train_items_used.reindex(re_hash.index) == rebuilt.item_list.reindex(re_hash.index).str.len())
record('Training data only', 'n_train_items_used equals the rebuilt count', bool(n_used_ok.all()), f'{int((~n_used_ok).sum())} differ')
leaks = sum(bool(set(rebuilt.item_list[u]) & held_out.get(u, set())) for u in re_hash.index)
record('Training data only', "no input contains the user's own validation or test item", leaks == 0, f'{leaks} users')
tmpl = META.get('prompt_template', '')
record('Training data only', 'prompt template uses product metadata and the user\'s own ratings only',
       bool(tmpl) and 'Reviews' not in META.get('item_text_fields_kept', []) and set(META.get('item_text_fields_kept', [])) <= {'Title', 'Brand', 'Category', 'Price'},
       META.get('item_text_fields_kept'))

[PASS] Training data only: frozen data files match the manifest (ok)
[PASS] Training data only: the run recorded every expected input file with the frozen hash ({'interactions.parquet': '298cab581ac9', 'sequences.parquet': '0a55f2c375c0', 'item_text.parquet': '39dbd1ac817d'})
[PASS] Training data only: every profiled user has training rows (0 without)
[PASS] Training data only: rebuilt input hash equals the stored input_sha256 for every profile (56,017/56,017 match)
[PASS] Training data only: n_train_items_used equals the rebuilt count (0 differ)
[PASS] Training data only: no input contains the user's own validation or test item (0 users)
[PASS] Training data only: prompt template uses product metadata and the user's own ratings only (['Title', 'Brand', 'Category', 'Price'])


## 5. Coverage, empty and duplicate profiles

In [8]:
seq_users = pd.read_parquet(DATA_DIR / 'sequences.parquet', columns=['user_idx', 'n_train'])
if MODE == 'full':
    expected = set(seq_users.user_idx.astype(int))
    if os.environ.get('AUDIT_EXPECTED_HASHES_JSON') is None:
        record('Coverage', f'all {N_USERS_FULL:,} users expected', len(expected) == N_USERS_FULL, len(expected))
else:                                                                    # rebuild the pilot's user list
    rng = np.random.default_rng(SEED)
    picks = []
    for lo, hi, n in PILOT_STRATA:
        pool = seq_users[(seq_users.n_train >= lo) & ((seq_users.n_train <= hi) if hi is not None else True)].user_idx.sort_values().to_numpy()
        picks.append(np.sort(rng.choice(pool, size=n, replace=False)))
    expected = set(np.concatenate(picks).astype(int).tolist())
got = set(p.index.astype(int))
record('Coverage', 'profiles + failed users = every expected user', got | failed_ids == expected,
       f'{len(got):,} profiles + {len(failed_ids)} failed = {len(got | failed_ids):,} of {len(expected):,} expected; '
       f'{len(expected - got - failed_ids)} missing, {len((got | failed_ids) - expected)} unexpected')
record('Coverage', 'no user is both profiled and failed', not (got & failed_ids), len(got & failed_ids))
record('Coverage', 'no failed users', len(failed_ids) == 0, len(failed_ids), warn_only=True)

text = profiles.profile_text.fillna('').astype(str)
words = text.str.split().str.len()
record('Empty / duplicate', 'no empty profiles', int((text.str.strip() == '').sum()) == 0, int((text.str.strip() == '').sum()))
record('Empty / duplicate', 'no very short profiles (under 10 words)', int((words < 10).sum()) == 0, int((words < 10).sum()), warn_only=True)
dup = profiles.assign(t=text)[text.duplicated(keep=False)]
dup_diff_input = dup.groupby('t').input_sha256.nunique().gt(1).sum() if len(dup) else 0
record('Empty / duplicate', 'identical profile texts only for users with identical inputs', int(dup_diff_input) == 0,
       f'{len(dup)} users share a text; {int(dup_diff_input)} texts shared across different inputs', warn_only=True)

NEAR = {'pairs_above_threshold': None}
if 'minilm' in VEC:
    E = VEC['minilm']; n = len(E); best = np.full(n, -1.0, np.float32); arg = np.zeros(n, np.int64)
    for s in range(0, n, 1024):
        S = E[s:s + 1024] @ E.T
        S[np.arange(len(S)), np.arange(s, s + len(S))] = -1
        arg[s:s + 1024] = S.argmax(1); best[s:s + 1024] = S.max(1)
    hs = profiles.input_sha256.to_numpy()
    near = (best >= NEAR_DUP_COSINE) & (hs != hs[arg])
    NEAR = {'threshold': NEAR_DUP_COSINE, 'users_with_near_duplicate': int(near.sum()),
            'median_nearest_cosine': round(float(np.median(best)), 4),
            'examples': [{'user_a': int(profiles.user_idx[i]), 'user_b': int(profiles.user_idx[arg[i]]), 'cosine': round(float(best[i]), 4)}
                         for i in np.flatnonzero(near)[:5]]}
    record('Empty / duplicate', f'near-duplicate vectors (MiniLM cosine >= {NEAR_DUP_COSINE}) between users with different inputs',
           NEAR['users_with_near_duplicate'] <= 0.01 * n, f"{NEAR['users_with_near_duplicate']} users; median nearest cosine {NEAR['median_nearest_cosine']}",
           warn_only=True)

[PASS] Coverage: all 56,017 users expected (56017)
[PASS] Coverage: profiles + failed users = every expected user (56,017 profiles + 0 failed = 56,017 of 56,017 expected; 0 missing, 0 unexpected)
[PASS] Coverage: no user is both profiled and failed (0)
[PASS] Coverage: no failed users (0)
[PASS] Empty / duplicate: no empty profiles (0)
[PASS] Empty / duplicate: no very short profiles (under 10 words) (0)
[PASS] Empty / duplicate: identical profile texts only for users with identical inputs (21 users share a text; 0 texts shared across different inputs)
[PASS] Empty / duplicate: near-duplicate vectors (MiniLM cosine >= 0.98) between users with different inputs (25 users; median nearest cosine 0.8406)


## 6. Text leakage check (brands from held-out items)
For each user: brands that appear **only** in their held-out (validation/test) items, not in their training items.
How often does the profile name one of them? The baseline does the same with a **random other user's** held-out
brands. Since profiles are built from training items only, the two rates should be about equal; a clearly higher
rate would point to leakage. Brands that are ordinary words (e.g. "Simple", "Pure") are skipped.

In [9]:
it = pd.read_parquet(DATA_DIR / 'item_text.parquet', columns=['item_idx', 'brand', 'title'])
brand_of = dict(zip(it.item_idx, it.brand.fillna('').astype(str).str.strip().str.lower()))
titles_low = it.title.fillna('').str.lower()
cand_brands = {b for b in set(brand_of.values()) if len(b) >= 4}
brand_count = pd.Series(list(brand_of.values())).value_counts()
title_df = pd.Series([w for t in titles_low for w in set(re.findall(r"[a-z0-9']+", t))]).value_counts()
generic = {b for b in cand_brands                # one-word brands that are ordinary words in many other titles
           if re.fullmatch(r"[a-z0-9']+", b) and int(title_df.get(b, 0)) > 3 * int(brand_count.get(b, 0)) + 20}
usable = cand_brands - generic
users = list(p.index)
rng2 = random.Random(SEED)
own_hits = base_hits = n_users_with = 0
own_examples = []
for u in users:
    tr = {brand_of[i] for i in rebuilt.item_list.get(u, [])}
    other = rng2.choice(users)
    def ho_brands(v):
        return ({brand_of[i] for i in held_out.get(v, set())} - tr) & usable
    mine, base = ho_brands(u), ho_brands(other) if other != u else set()
    t = p.profile_text[u].lower()
    if mine:
        n_users_with += 1
        if any(re.search(rf'\b{re.escape(b)}\b', t) for b in mine):
            own_hits += 1
            if len(own_examples) < 5:
                own_examples.append({'user_idx': int(u), 'brands': sorted(mine)[:3]})
    if base and any(re.search(rf'\b{re.escape(b)}\b', t) for b in base):
        base_hits += 1
LEAK = {'users_checked': len(users), 'users_with_held_out_only_brands': n_users_with,
        'profiles_naming_own_held_out_brand': own_hits, 'own_rate': round(own_hits / max(1, len(users)), 4),
        'baseline_naming_random_users_held_out_brand': base_hits, 'baseline_rate': round(base_hits / max(1, len(users)), 4),
        'generic_brands_skipped': len(generic), 'examples': own_examples}
print(json.dumps(LEAK, indent=1))
clearly_higher = LEAK['own_rate'] > 2 * LEAK['baseline_rate'] + 0.005
record('Text leakage', "profiles do not name held-out-only brands more often than chance", not clearly_higher,
       f"own {LEAK['own_rate']:.2%} vs baseline {LEAK['baseline_rate']:.2%}", warn_only=True)

{
 "users_checked": 56017,
 "users_with_held_out_only_brands": 51850,
 "profiles_naming_own_held_out_brand": 266,
 "own_rate": 0.0047,
 "baseline_naming_random_users_held_out_brand": 34,
 "baseline_rate": 0.0006,
 "generic_brands_skipped": 35,
 "examples": [
  {
   "user_idx": 532,
   "brands": [
    "eltamd"
   ]
  },
  {
   "user_idx": 589,
   "brands": [
    "derma-e",
    "l'or\u00e9al paris"
   ]
  },
  {
   "user_idx": 896,
   "brands": [
    "l'oreal paris",
    "l'or\u00e9al paris"
   ]
  },
  {
   "user_idx": 1057,
   "brands": [
    "bondi sands",
    "l'oreal paris"
   ]
  },
  {
   "user_idx": 1200,
   "brands": [
    "blistex",
    "curel"
   ]
  }
 ]
}
[PASS] Text leakage: profiles do not name held-out-only brands more often than chance (own 0.47% vs baseline 0.06%)


## 7. Do the stored vectors belong to their texts?
A sample of profiles is re-embedded with the same encoders and commits. Each stored vector must be very close to
its own re-embedded text (cosine ≥ 0.99; tiny differences come from CPU versus GPU) and must be the nearest one
in the sample, which catches rows shifted between users.

In [10]:
from sentence_transformers import SentenceTransformer
rng3 = np.random.default_rng(SEED)
idx = np.sort(rng3.choice(len(profiles), size=min(REEMBED_SAMPLE, len(profiles)), replace=False))
REEMBED = {}
ENC_META = META.get('encoders', {}) if isinstance(META.get('encoders'), dict) else {}
for name in ENCODER_IDS:                                   # both encoders are required; none is skipped
    enc_meta = ENC_META.get(name)
    if not isinstance(enc_meta, dict) or name not in VEC:
        record('Vectors match texts', f'{name}: encoder record and vectors present', False,
               'no encoder record in profiles_meta.json' if not isinstance(enc_meta, dict) else 'vectors did not load')
        continue
    pinned = enc_meta.get('model') == ENCODER_IDS[name] and enc_meta.get('revision') == ENCODER_REVISIONS[name]
    record('Vectors match texts', f'{name}: encoder record names the pinned model and commit', pinned,
           f"{enc_meta.get('model')}@{str(enc_meta.get('revision'))[:12]}")
    model_id = ENCODER_IDS[name]
    local = Path(model_id).exists()
    st = SentenceTransformer(model_id, revision=None if local else ENCODER_REVISIONS[name], device='cpu')
    new = st.encode([enc_meta.get('prefix', '') + t for t in text.iloc[idx]], normalize_embeddings=True,
                    convert_to_numpy=True).astype(np.float32)
    S = new @ VEC[name][idx].T
    own = np.diag(S)
    self_nearest = float((own >= S.max(1) - 1e-6).mean())        # ties (identical texts) count as a match
    REEMBED[name] = {'model': model_id, 'revision': None if local else enc_meta['revision'], 'sample': int(len(idx)),
                     'min_cosine': round(float(own.min()), 5), 'mean_cosine': round(float(own.mean()), 5),
                     'self_is_nearest': round(self_nearest, 4)}
    record('Vectors match texts', f'{name}: re-embedded sample matches stored vectors',
           own.min() >= 0.99 and self_nearest >= 0.99, REEMBED[name])
    del st

[PASS] Vectors match texts: minilm: encoder record names the pinned model and commit (sentence-transformers/all-MiniLM-L6-v2@1110a243fdf4)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (20 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

[PASS] Vectors match texts: e5_large: re-embedded sample matches stored vectors ({'model': 'intfloat/multilingual-e5-large', 'revision': '3d7cfbdacd47fdda877c5cd8a79fbcc4f2a574f3', 'sample': 100, 'min_cosine': 1.0, 'mean_cosine': 1.0, 'self_is_nearest': 1.0})


## 8. Settings and generation hygiene

In [11]:
teacher_ok = (META.get('teacher_revision') == EXPECTED_TEACHER_REVISION and META.get('teacher_model') == EXPECTED_TEACHER)
TEMPLATE_SHA = hashlib.sha256(str(META.get('prompt_template', '')).encode()).hexdigest()
record('Settings', 'approved prompt (v2): the stored template itself hashes to the approved hash',
       TEMPLATE_SHA == EXPECTED_PROMPT_SHA256 and META.get('prompt_sha256') == EXPECTED_PROMPT_SHA256,
       f'template hashes to {TEMPLATE_SHA[:16]}, stored {str(META.get("prompt_sha256"))[:16]}, approved {EXPECTED_PROMPT_SHA256[:16]}')
record('Settings', 'pinned teacher model and commit', teacher_ok, f"{META.get('teacher_model')}@{str(META.get('teacher_revision'))[:12]}")
record('Settings', 'thinking off', META.get('enable_thinking') is False, META.get('enable_thinking'))
EXPECTED_TEACHER_TAG = os.environ.get('AUDIT_EXPECTED_TEACHER_TAG') or f'{EXPECTED_TEACHER}@{EXPECTED_TEACHER_REVISION}'
wrong_tag = profiles.teacher_model.astype(str) != EXPECTED_TEACHER_TAG
record('Settings', 'every profile row names the approved teacher and commit', not wrong_tag.any(),
       f'{int(wrong_tag.sum())} rows differ, e.g. {profiles.teacher_model[wrong_tag].unique()[:3].tolist()}' if wrong_tag.any()
       else EXPECTED_TEACHER_TAG)

THINK = text.str.contains(r'<think>|</think>', regex=True)
METATXT = text.str.contains(r"\blet me\b|\bokay,|\bfirst,? i\b|\bthe user\b|\bi (?:need|will|should)\b", case=False, regex=True)
SENS = re.compile(r"\b(she|he|her|his|woman|women|man|men|female|male|girl|boy|teen\w*|pregnan\w*|wife|husband|"
                  r"diabet\w*|eczema|psoriasis|rosacea|cancer|disease)\b", re.I)
sens_users = []
titles_by_item = dict(zip(it.item_idx, titles_low))
for u, t in zip(profiles.user_idx, text):
    found = {m.lower() for m in SENS.findall(t)}
    if found:
        blob = ' '.join(titles_by_item.get(i, '') for i in rebuilt.item_list.get(u, []))
        if any(not re.search(rf'\b{re.escape(w)}\b', blob) for w in found):
            sens_users.append(int(u))
HYGIENE = {'thinking_text': int(THINK.sum()), 'reasoning_or_meta_text': int(METATXT.sum()),
           'over_120_words': int((words > 120).sum()), 'possible_sensitive_terms': len(sens_users),
           'sensitive_examples': sens_users[:10], 'mean_words': round(float(words.mean()), 1),
           'median_words': float(words.median()), 'max_words': int(words.max())}
record('Hygiene', 'no thinking text', HYGIENE['thinking_text'] == 0, HYGIENE['thinking_text'], warn_only=True)
record('Hygiene', 'no reasoning or meta text', HYGIENE['reasoning_or_meta_text'] == 0, HYGIENE['reasoning_or_meta_text'], warn_only=True)
record('Hygiene', 'no profile over 120 words', HYGIENE['over_120_words'] == 0, HYGIENE['over_120_words'], warn_only=True)
record('Hygiene', 'possible sensitive terms (read these by hand)', HYGIENE['possible_sensitive_terms'] <= 0.005 * len(profiles),
       f"{HYGIENE['possible_sensitive_terms']} profiles, e.g. users {sens_users[:5]}", warn_only=True)

[PASS] Settings: approved prompt (v2): the stored template itself hashes to the approved hash (template hashes to a18472f1aefb1a5f, stored a18472f1aefb1a5f, approved a18472f1aefb1a5f)
[PASS] Settings: pinned teacher model and commit (Qwen/Qwen3.5-9B@c20223623576)
[PASS] Settings: thinking off (False)
[PASS] Settings: every profile row names the approved teacher and commit (Qwen/Qwen3.5-9B@c202236235762e1c871ad0ccb60c8ee5ba337b9a)
[PASS] Hygiene: no thinking text (0)
[WARN] Hygiene: no reasoning or meta text (33)
[PASS] Hygiene: no profile over 120 words (0)
[PASS] Hygiene: possible sensitive terms (read these by hand) (43 profiles, e.g. users [1197, 1307, 1575, 2422, 3698])


## 9. Chunks (full run only): do the early per-part files agree with the final files?

In [12]:
CHUNKS = None
chunk_dir = RUN_DIR / 'chunks'
parts = sorted(chunk_dir.glob('chunk_*.parquet')) if chunk_dir.exists() else []
parts = [c for c in parts if re.fullmatch(r'chunk_\d{5}\.parquet', c.name)]
if MODE == 'full' and parts:
    final_pos = {int(u): r for r, u in enumerate(profiles.user_idx)}
    mism, users_in_chunks, setups = 0, 0, set()
    for cp in parts:
        setups |= set(pd.read_parquet(cp, columns=['setup_sha']).setup_sha)
        for name in VEC:
            with np.load(cp.with_name(cp.stem + f'_{name}.npz'), allow_pickle=False) as z:
                cu, ce = z['user_idx'], z['embeddings']
            rows = [final_pos.get(int(u)) for u in cu]
            if any(r is None for r in rows) or not np.allclose(VEC[name][rows], ce, atol=1e-6):
                mism += 1
            if name == 'minilm':
                users_in_chunks += len(cu)
    CHUNKS = {'parts': len(parts), 'users_in_part_files': users_in_chunks, 'mismatching_part_files': mism,
              'setups': len(setups)}
    record('Chunks', 'per-part vectors equal the final vectors', mism == 0, CHUNKS)
    record('Chunks', 'all parts made with one setup', len(setups) == 1, len(setups))
else:
    print('Not a full run, or no chunks: skipped.')

[PASS] Chunks: per-part vectors equal the final vectors ({'parts': 29, 'users_in_part_files': 56017, 'mismatching_part_files': 0, 'setups': 1})
[PASS] Chunks: all parts made with one setup (1)


## 10. Verdict and report

In [13]:
res = pd.DataFrame(RESULTS)
n_fail, n_warn = int((res.status == 'FAIL').sum()), int((res.status == 'WARN').sum())
VERDICT = 'FAIL' if n_fail else ('PASS with warnings' if n_warn else 'PASS')
REPORT = {'run': RUN_NAME, 'mode': MODE, 'verdict': VERDICT, 'n_fail': n_fail, 'n_warn': n_warn,
          'audited_utc': datetime.datetime.now(datetime.timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ'),
          'n_profiles': int(len(profiles)), 'n_failed_users': len(failed_ids), 'checks': RESULTS,
          'near_duplicates': NEAR, 'text_leakage': LEAK, 'reembed': REEMBED, 'hygiene': HYGIENE, 'chunks': CHUNKS,
          'teacher': f"{META.get('teacher_model')}@{META.get('teacher_revision')}",
          'versions': {n: _ver(n) for n in ('numpy', 'pandas', 'pyarrow', 'sentence-transformers', 'transformers')}}
(AUDIT_DIR / 'audit_report.json').write_text(json.dumps(REPORT, indent=2, default=str))
lines = [f'# Profile audit: {RUN_NAME}', '', f"**Verdict: {VERDICT}** ({n_fail} failed, {n_warn} warnings, "
         f"{int((res.status == 'PASS').sum())} passed). {len(profiles):,} profiles, {len(failed_ids)} failed users.", '',
         '| Group | Check | Result | Detail |', '|---|---|---|---|']
lines += [f"| {r['group']} | {r['check']} | {r['status']} | {r['detail'].replace('|', '/')[:160]} |" for r in RESULTS]
(AUDIT_DIR / 'audit_summary.md').write_text('\n'.join(lines) + '\n')
print('\n'.join(lines[:3]))
print('\nSaved', AUDIT_DIR / 'audit_report.json', 'and audit_summary.md')
if n_fail:
    print('\nFAILED checks:'); print(res[res.status == 'FAIL'][['group', 'check', 'detail']].to_string(index=False))
if n_warn:
    print('\nWarnings to look at:'); print(res[res.status == 'WARN'][['group', 'check', 'detail']].to_string(index=False))
if IN_COLAB:
    from google.colab import drive
    drive.flush_and_unmount()

# Profile audit: full_v2_qwen35_9b

**Verdict: PASS with warnings** (0 failed, 1 warnings, 39 passed). 56,017 profiles, 0 failed users.

Saved /content/drive/.shortcut-targets-by-id/1K2gvVTjU-77GWGMI14ixFt5F57IaijZB/298A/model1_profiles/audits/full_v2_qwen35_9b/audit_report.json and audit_summary.md

Warnings to look at:
  group                     check detail
Hygiene no reasoning or meta text     33
